In [ ]:
# !git clone https://github.com/uvavision/RerankingTransformer.git


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# !pwd
# !ls

In [ ]:
# !find /content/drive/MyDrive -type d -iname "*rerank*" 2>/dev/null

In [ ]:
# !mv /content/RerankingTransformer "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/"

In [ ]:
from pathlib import Path

repo = next(Path("/content/drive/MyDrive").rglob("RerankingTransformer"))
print(repo)

In [ ]:
# %cd "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/RerankingTransformer"

RerankingTransformers (RRTs): Experiments on Stanford Online Products
https://github.com/uvavision/RerankingTransformer/tree/main/RRT_SOP

Preparation

In [ ]:
!pip install sacred

In [ ]:

# %cd "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/RerankingTransformer/RRT_SOP"
# !python prepare_data.py

ô trên tôi đã dừng khi đang chạy

folder đã xuất hiện đầy đủ trên Google Drive web, không còn trạng thái upload?

cell dưới không lỗi đâu, vì tôi dừng giữa chừng nên nó hiện đỏ á

In [ ]:
# from pathlib import Path
# root = Path("/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/RerankingTransformer/RRT_SOP/data/Stanford_Online_Products")

# for folder in ["bicycle_final", "chair_final",  "kettle_final"]:
#   p = root / folder
#   files = list(p.glob("*")) if p.exists() else []
#   print(folder,  "exists =", p.exists(), "| files = ", len(files))

dùng luôn 2 category hiện có:

bicycle_final: 8313 ảnh
kettle_final: 9510 ảnh

rủi ro chủ yếu nằm ở ý nghĩa kết quả, không phải ở việc code có chạy hay không.

mục tiêu hiện tại chỉ là:

kiểm tra code đọc được ảnh → load checkpoint → chạy global retrieval → chạy rerank → xuất Recall@K

tạo một bộ test tạm.
đang code: tạo train_subset.txt và test_subset.txt trong SOP_subset_smoketest, nhưng ảnh vẫn giữ nguyên ở Stanford_Online_Products/.

Code lọc ra chỉ những dòng có:

bicycle_final/ và
kettle_final/

bây giờ sẽ làm code kiểm tra file ảnh có thực sự tồn tại;
chỉ ghi những sample hợp lệ vào train_subset.txt / test_subset.txt.(đang làm, chưa xong)

label được đổi từ 1-based sang 0-based bằng class_id - 1

In [ ]:
from pathlib import Path
base = Path("/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/RerankingTransformer/RRT_SOP/data")

meta_dir = base / "SOP_subset_smoketest"
img_root = base / "Stanford_Online_Products"

categories = ("bicycle_final/", "kettle_final/")


for src_name, dst_name in [
    ("Ebay_train.txt", "train_subset.txt"),
    ("Ebay_test.txt", "test_subset.txt"),
]:
    src = meta_dir /src_name
    dst = meta_dir /dst_name

    lines = src.read_text().splitlines()

    category_lines = [
        line for line in lines[1:]
        if any(cate in line for cate in categories )
    ]

    valid = []
    missing = []

    for line in category_lines:
      parts = line.split()

      img_rel_path = parts[-1]
      img_path = img_root / img_rel_path



      if img_path.exists():
        class_id = int(parts[1]) - 1

        rel_from_subset = Path("../Stanford_Online_Products") / img_rel_path
        valid.append(f"{rel_from_subset.as_posix()},{class_id}")
      else:
        missing.append(img_rel_path)

    dst.write_text("\n".join(valid) + "\n")


    print(f"\n{dst_name}")
    print("Metadata thuộc 2 category :", len(category_lines))
    print("Ảnh thực sự tồn tại      :", len(valid))
    print("Ảnh còn thiếu             :", len(missing))

Bám vào
https://github.com/uvavision/RerankingTransformer/tree/main/RRT_SOP


Clone visdom_logger vào /content trước

In [ ]:
# %cd /content
# !git clone https://github.com/luizgh/visdom_logger.git

move nó vào cùng khu vực Drive của project

In [ ]:
# !mv /content/visdom_logger \
# "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/"

Rồi cài từ Drive:

In [ ]:
!pip install -e \
"/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/visdom_logger"

ModuleNotFoundError: No module named 'faiss'

In [ ]:
!pip install faiss-cpu

In [ ]:
from pathlib import Path

resnet_file = Path(
    "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/"
    "Code/Reranking/RerankingTransformer/RRT_SOP/"
    "models/architectures/resnet.py"
)

text = resnet_file.read_text()

text = text.replace(
    "from torchvision.models.resnet import conv1x1, BasicBlock, Bottleneck, model_urls",
    "from torchvision.models.resnet import conv1x1, BasicBlock, Bottleneck"
)

text = text.replace(
    "from torchvision.models.utils import load_state_dict_from_url",
    "from torch.hub import load_state_dict_from_url"
)

# URL pretrained ResNet50 tương ứng API torchvision cũ
insert = """
model_urls = {
    'resnet50': 'https://download.pytorch.org/models/resnet50-0676ba61.pth',
}
"""

if "model_urls = {" not in text:
    text = text.replace(
        "from copy import deepcopy",
        "from copy import deepcopy\n" + insert
    )

resnet_file.write_text(text)

print("Đã patch:", resnet_file)

repo 2021 dùng API cũ của torchvision, còn Colab hiện tại dùng torchvision mới nên:ImportError: cannot import name 'model_urls' from 'torchvision.models.resnet'.      cell này một lần để vá file resnet.py cho torchvision mới

In [ ]:
from pathlib import Path

resnet_file = Path(
    "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/"
    "Code/Reranking/RerankingTransformer/RRT_SOP/"
    "models/architectures/resnet.py"
)

text = resnet_file.read_text()

text = text.replace(
    "from torchvision.models.resnet import conv1x1, BasicBlock, Bottleneck, model_urls",
    "from torchvision.models.resnet import conv1x1, BasicBlock, Bottleneck"
)

text = text.replace(
    "from torchvision.models.utils import load_state_dict_from_url",
    "from torch.hub import load_state_dict_from_url"
)

# URL pretrained ResNet50 tương ứng API torchvision cũ
insert = """
model_urls = {
    'resnet50': 'https://download.pytorch.org/models/resnet50-0676ba61.pth',
}
"""

if "model_urls = {" not in text:
    text = text.replace(
        "from copy import deepcopy",
        "from copy import deepcopy\n" + insert
    )

resnet_file.write_text(text)

print("Đã patch:", resnet_file)

Với subset, loaders.num_classes = 1523, nhưng checkpoint full SOP có classifier cho 11318 lớp. Vì vậy ta ép model được dựng với 11318 lớp để khớp checkpoint.

In [ ]:
from pathlib import Path

eval_file = Path(
    "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/"
    "Code/Reranking/RerankingTransformer/RRT_SOP/eval_global.py"
)

text = eval_file.read_text()

old = "model = get_model(num_classes=loaders.num_classes)"
new = "model = get_model(num_classes=11318)"

if old in text:
    text = text.replace(old, new)
    eval_file.write_text(text)
    print("Đã patch eval_global.py")
else:
    print("Không tìm thấy dòng cần patch hoặc file đã được patch trước đó.")

override cấu hình Sacred ngay khi chạy eval_global.py để nó đọc subset của mình.

In [ ]:
%cd "/content/drive/MyDrive/VIR/PROJECT_VIR_23127008_23127057/Code/Reranking/RerankingTransformer/RRT_SOP"

!python eval_global.py \
-F logs/eval_global_subset \
with \
temp_dir=logs/eval_global_subset \
resume=rrt_sop_ckpts/rrt_r50_sop_global.pt \
dataset.sop_global \
model.resnet50 \
dataset.data_path=data/SOP_subset_smoketest \
dataset.train_file=train_subset.txt \
dataset.test_file=test_subset.txt \
dataset.test_batch_size=128 \
dataset.num_workers=2